# Question 2 — LoRA vs QLoRA vs DoRA vs AdaLoRA for function calling
Zaid Bin Naveed · 23K-0028 · Agentic AI (AI4015)

**Run on Google Colab with a GPU runtime (T4 or better):** Runtime → Change runtime type → GPU, then Run all.
Every run writes its metrics to `results/<run>.json`, so if the session disconnects, re-running skips finished runs.
Set `USE_DRIVE = True` to keep `results/` on Google Drive across sessions. When finished, copy `results/` into
`Q2_LoRA_Comparison/results/` in the repo and run `python Q2_LoRA_Comparison/build_report.py`.

| Run | Method | Base precision | Rank |
|---|---|---|---|
| 0 | Base model (no tuning) | bf16 / fp16 | – |
| 1 | LoRA | bf16 / fp16 | r = 8 |
| 2 | QLoRA | 4-bit NF4 | r = 8 |
| 3 | DoRA | bf16 / fp16 | r = 8 |
| 4 | AdaLoRA | bf16 / fp16 | r 12 → 8 |

In [ ]:
%pip install -q "transformers==4.57.6" "peft==0.17.1" "accelerate==1.10.1" "datasets==4.0.0" "bitsandbytes>=0.46,<0.51" "matplotlib" "pandas"

In [ ]:
import gc
import json
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import peft
import torch
import transformers
from datasets import load_dataset
from peft import AdaLoraConfig, LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainerCallback,
                          TrainingArguments, set_seed)

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
RESULTS = Path("/content/drive/MyDrive/q2_results" if USE_DRIVE else "results")
RESULTS.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "glaiveai/glaive-function-calling-v2"
SEED = 42
N_TOTAL, N_TEST = 2000, 200
MAX_LEN = 768
EPOCHS = 2
LR = 2e-4
BATCH, GRAD_ACC = 4, 4
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_ALPHA, LORA_DROPOUT = 16, 0.05
MAX_NEW_TOKENS = 192
EVAL_BATCH = 16

assert torch.cuda.is_available(), "Switch the Colab runtime to a GPU."
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
GPU = torch.cuda.get_device_name(0)
ENV = {"gpu": GPU, "dtype": str(DTYPE), "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__}
print(ENV)

## Data: glaive-function-calling-v2 → (user message → one function call JSON)
Each row has a `system` field with the JSON schemas of the available functions and a `chat` transcript.
I keep rows whose **first** assistant turn is a `<functioncall>`, parse the call into
`{"name": ..., "arguments": {...}}`, and drop rows whose arguments are not valid JSON or whose prompt plus answer exceeds `MAX_LEN` tokens.
The rows are shuffled with a fixed seed, and the first 2,000 usable ones are kept: 200 test rows held out up front, 1,800 for training.

In [ ]:
CALL_RE = re.compile(r'"name":\s*"([^"]+)",\s*"arguments":\s*\'(.*)\'\s*}', re.S)
TURN_RE = re.compile(r"USER:\s*(.*?)\s*ASSISTANT:\s*(.*?)(?:<\|endoftext\|>|\n\n\nUSER:|\n\n\nFUNCTION RESPONSE:|$)", re.S)
SYSTEM_PROMPT = (
    "You are a customer-support agent that acts by calling functions. Available functions:\n{functions}\n"
    'Reply with only a JSON object of the form {{"name": "<function name>", "arguments": {{...}}}} and nothing else.'
)


def parse_functions(system):
    decoder, functions, i = json.JSONDecoder(), [], system.find("{")
    while 0 <= i < len(system):
        try:
            obj, end = decoder.raw_decode(system, i)
        except json.JSONDecodeError:
            return []
        functions.append(obj)
        i = system.find("{", end)
    return functions


def parse_row(row):
    functions = parse_functions(row["system"])
    turn = TURN_RE.search(row["chat"])
    if not functions or not turn or not turn.group(2).startswith("<functioncall>"):
        return None
    call = CALL_RE.search(turn.group(2))
    if not call:
        return None
    try:
        arguments = json.loads(call.group(2))
    except json.JSONDecodeError:
        return None
    names = [f.get("name") for f in functions]
    if call.group(1) not in names or not isinstance(arguments, dict):
        return None
    return {"user": turn.group(1).strip(), "functions": functions, "names": names,
            "target": {"name": call.group(1), "arguments": arguments}}


def messages(ex):
    return [{"role": "system", "content": SYSTEM_PROMPT.format(functions=json.dumps(ex["functions"], indent=1))},
            {"role": "user", "content": ex["user"]}]


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def encode(ex):
    prompt = tokenizer.apply_chat_template(messages(ex), tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(json.dumps(ex["target"]) + "<|im_end|>", add_special_tokens=False)["input_ids"]
    return {"input_ids": prompt_ids + answer_ids, "labels": [-100] * len(prompt_ids) + answer_ids}


raw = load_dataset(DATASET_ID, split="train").shuffle(seed=SEED)
examples = []
for idx, row in enumerate(raw):
    ex = parse_row(row)
    if ex and len(encode(ex)["input_ids"]) <= MAX_LEN:
        ex["source_index"] = idx
        examples.append(ex)
    if len(examples) == N_TOTAL:
        break

test_set, train_set = examples[:N_TEST], examples[N_TEST:]
train_data = [encode(ex) for ex in train_set]
(RESULTS / "data_split.json").write_text(json.dumps(
    {"seed": SEED, "rows_scanned": idx + 1, "test_source_indices": [e["source_index"] for e in test_set],
     "train_source_indices": [e["source_index"] for e in train_set]}))
print(f"scanned {idx + 1} rows -> {len(train_set)} train / {len(test_set)} test")
print("distinct gold functions in test:", len({e['target']['name'] for e in test_set}))
print(json.dumps({"user": test_set[0]["user"], "target": test_set[0]["target"]}, indent=1))

## Metrics
* **JSON validity**: `json.loads(output.strip())` succeeds and gives an object. Strict: any text around the JSON counts as invalid.
* **Tool accuracy**: the first JSON object found in the output has the gold function `name`.
* **Full match**: correct name and the `arguments` dict is exactly equal to the gold one.
* **Hallucinated tool**: the predicted name is not among the functions offered in the prompt (an unauthorised action).
* **Wrong action**: the model calls a real, offered function, but the gold answer is a different one (e.g. a refund when the user asked about delivery). Every test example expects a call, so there is no "should not have called" case.
* **No call / refusal**: no function name could be extracted although a call was expected.
* **Inference speed**: new tokens per second and mean milliseconds per example, measured with greedy batched decoding (batch 16), once with the adapter **unmerged** and once **merged** into a fresh 16-bit copy of the base model. Unmerged DoRA recomputes a weight norm on every forward pass, so it is slower until merged. A QLoRA adapter can only be merged after de-quantising the base to fp16/bf16, so the merged QLoRA model is no longer the NF4 model it was trained on. Its merged accuracy is reported so that drift is visible.

Every miss also gets an automatic error label for the error analysis.

In [ ]:
def first_json_object(text):
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, _ = decoder.raw_decode(text, i)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict):
                return obj
    return None


def score(output, ex):
    try:
        valid = isinstance(json.loads(output.strip()), dict)
    except json.JSONDecodeError:
        valid = False
    obj = first_json_object(output) or {}
    name, args = obj.get("name"), obj.get("arguments")
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            pass
    gold = ex["target"]
    tool_ok = name == gold["name"]
    full = tool_ok and args == gold["arguments"]

    if full:
        label = "correct" if valid else "correct call wrapped in extra text"
    elif not obj:
        label = "broken JSON" if "{" in output else "refused / no tool call"
    elif name not in ex["names"]:
        label = "hallucinated tool"
    elif not tool_ok:
        label = "wrong tool"
    elif not isinstance(args, dict):
        label = "broken arguments"
    elif set(args) - set(gold["arguments"]):
        label = "made-up argument"
    elif set(gold["arguments"]) - set(args):
        label = "missing argument"
    else:
        label = "wrong argument value"
    return {"json_valid": valid, "tool_correct": tool_ok, "full_match": full,
            "hallucinated_tool": bool(obj) and name not in ex["names"],
            "wrong_action": bool(obj) and name in ex["names"] and not tool_ok,
            "no_call": not name, "label": label}


@torch.no_grad()
def generate_all(model, dataset):
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    outputs, new_tokens = [], 0
    torch.cuda.synchronize()
    start = time.perf_counter()
    for i in range(0, len(dataset), EVAL_BATCH):
        prompts = [tokenizer.apply_chat_template(messages(ex), tokenize=False, add_generation_prompt=True)
                   for ex in dataset[i:i + EVAL_BATCH]]
        batch = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        gen = gen[:, batch["input_ids"].shape[1]:]
        new_tokens += int((gen != tokenizer.pad_token_id).sum())
        outputs += tokenizer.batch_decode(gen, skip_special_tokens=True)
    torch.cuda.synchronize()
    return outputs, time.perf_counter() - start, new_tokens


def evaluate(model, run_name):
    outputs, seconds, new_tokens = generate_all(model, test_set)
    scored = [score(o, ex) | {"user": ex["user"], "gold": ex["target"], "output": o} for o, ex in zip(outputs, test_set)]
    with open(RESULTS / f"predictions_{run_name}.jsonl", "w") as fh:
        fh.writelines(json.dumps(s) + "\n" for s in scored)
    n = len(scored)
    counts = {key: sum(s[key] for s in scored)
              for key in ("json_valid", "tool_correct", "full_match", "hallucinated_tool", "wrong_action", "no_call")}
    return {"n_test": n, **counts,
            "json_valid_pct": 100 * counts["json_valid"] / n, "tool_accuracy_pct": 100 * counts["tool_correct"] / n,
            "full_match_pct": 100 * counts["full_match"] / n, "wrong_action_pct": 100 * counts["wrong_action"] / n,
            "eval_seconds": seconds, "eval_ms_per_example": 1000 * seconds / n, "eval_new_tokens_per_s": new_tokens / seconds,
            "error_labels": pd.Series([s["label"] for s in scored]).value_counts().to_dict()}


def evaluate_merged(name):
    # Merging always goes into a fresh 16-bit copy of the base. For QLoRA that means the adapter, trained against
    # NF4-quantized weights, is folded into de-quantized fp16/bf16 weights it never saw during training.
    free_gpu()
    merged = PeftModel.from_pretrained(load_base(quantized=False), f"adapters/{name}").merge_and_unload()
    result = evaluate(merged, f"{name}_merged")
    del merged
    free_gpu()
    return result

## Models and adapter configs
All runs use the same target modules (all attention and MLP projections), α = 16, dropout 0.05, seed, data, epochs (2), LR (2e-4, cosine, 3% warm-up), effective batch 16 and gradient checkpointing.
On a T4 (no bf16), the base weights are fp16 and fp16 mixed precision is used. Trainable adapter weights are kept in fp32 so the AMP gradient scaler works.
AdaLoRA starts at r = 12 and prunes to an average budget of r = 8. The callback calls `update_and_allocate` after each optimiser step, which is how PEFT documents AdaLoRA training.

In [ ]:
def load_base(quantized):
    kwargs = {"dtype": DTYPE, "device_map": {"": 0}}
    if quantized:
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)


def steps_per_run():
    return EPOCHS * -(-len(train_data) // (BATCH * GRAD_ACC))


def adapter_config(method, r):
    common = dict(target_modules=TARGET_MODULES, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type="CAUSAL_LM")
    if method == "adalora":
        total = steps_per_run()
        return AdaLoraConfig(init_r=round(1.5 * r), target_r=r, tinit=int(0.1 * total), tfinal=int(0.3 * total),
                             deltaT=10, total_step=total, **common)
    return LoraConfig(r=r, use_dora=(method == "dora"), **common)


class AdaLoraBudget(TrainerCallback):
    def on_optimizer_step(self, args, state, control, model=None, **kwargs):
        model.base_model.update_and_allocate(state.global_step)


def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, value: seq + [value] * (width - len(seq))
    return {"input_ids": torch.tensor([pad(b["input_ids"], tokenizer.pad_token_id) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
            "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch])}


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
def run_experiment(method, r):
    name = f"{method}_r{r}"
    out = RESULTS / f"{name}.json"
    if out.exists():
        print(f"{name}: already done, loading")
        return json.loads(out.read_text())

    set_seed(SEED)
    free_gpu()
    model = load_base(quantized=(method == "qlora"))
    if method == "qlora":
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    model = get_peft_model(model, adapter_config(method, r))
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    trainable, total = model.get_nb_trainable_parameters()

    tokenizer.padding_side = "right"
    args = TrainingArguments(
        output_dir=f"checkpoints/{name}", num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine",
        warmup_ratio=0.03, per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
        fp16=not BF16, bf16=BF16, logging_steps=10, save_strategy="no", report_to="none", seed=SEED,
        data_seed=SEED, remove_unused_columns=False, dataloader_pin_memory=True)
    trainer = Trainer(model=model, args=args, train_dataset=train_data, data_collator=collate,
                      callbacks=[AdaLoraBudget()] if method == "adalora" else [])
    torch.cuda.synchronize()
    start = time.perf_counter()
    train_out = trainer.train()
    torch.cuda.synchronize()
    minutes = (time.perf_counter() - start) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3

    result = {"run": name, "method": method, "rank": r, "trainable_params": trainable, "total_params": total,
              "trainable_pct": 100 * trainable / total, "peak_train_mem_gb": peak_gb, "train_minutes": minutes,
              "train_loss": train_out.training_loss, "steps": trainer.state.global_step,
              "loss_curve": [h["loss"] for h in trainer.state.log_history if "loss" in h], **ENV}
    if method == "adalora":
        result["adalora_final_ranks"] = {k: int(sum(v)) for k, v in model.peft_config["default"].rank_pattern.items()} \
            if model.peft_config["default"].rank_pattern else None
    result |= evaluate(model, name)
    model.save_pretrained(f"adapters/{name}")
    del trainer, model
    result["merged"] = evaluate_merged(name)
    out.write_text(json.dumps(result, indent=2))
    print(json.dumps({k: v for k, v in result.items() if k not in ("loss_curve", "adalora_final_ranks", "error_labels", "merged")},
                     indent=1))
    print("merged:", {k: round(v, 2) for k, v in result["merged"].items() if k.endswith(("_pct", "_per_s", "_example"))})
    return result

## Run 0: base model without fine-tuning (the "before" picture)

In [ ]:
out = RESULTS / "base.json"
if out.exists():
    base_result = json.loads(out.read_text())
else:
    free_gpu()
    model = load_base(quantized=False)
    total = sum(p.numel() for p in model.parameters())
    base_result = {"run": "base", "method": "base", "rank": None, "trainable_params": 0, "total_params": total,
                   "trainable_pct": 0.0, "peak_train_mem_gb": None, "train_minutes": 0.0, **ENV}
    base_result |= evaluate(model, "base")
    base_result["peak_eval_mem_gb"] = torch.cuda.max_memory_allocated() / 1024**3
    out.write_text(json.dumps(base_result, indent=2))
    del model
    free_gpu()
print({k: base_result[k] for k in ("json_valid_pct", "tool_accuracy_pct", "full_match_pct", "error_labels")})

## Runs 1–4: the four adaptation methods (same everything except the method)

In [ ]:
METHODS = ["lora", "qlora", "dora", "adalora"]
main_results = [run_experiment(m, 8) for m in METHODS]

In [ ]:
COLUMNS = ["run", "trainable_params", "trainable_pct", "peak_train_mem_gb", "train_minutes", "json_valid", "tool_correct",
           "full_match", "hallucinated_tool", "wrong_action", "no_call", "eval_new_tokens_per_s", "eval_ms_per_example"]
table = pd.DataFrame([base_result, *main_results])[COLUMNS]
merged = [{"run": r["run"], **{k: r["merged"][k] for k in ("full_match", "tool_correct", "eval_new_tokens_per_s",
                                                         "eval_ms_per_example")}} for r in main_results]
table = table.merge(pd.DataFrame(merged), on="run", how="left", suffixes=("", "_merged"))
table.to_csv(RESULTS / "main_results.csv", index=False)
table.round(3)

## Section 3: is a bigger rank actually better?
The winner is chosen automatically by full-match count, then tool accuracy, then lower peak memory. Set `WINNER` by hand to override.
It is retrained at r = 4, 16 and 32, and the r = 8 run is reused.

In [ ]:
best = max(main_results, key=lambda r: (r["full_match"], r["tool_correct"], -r["peak_train_mem_gb"]))
WINNER = best["method"]
print("winner:", WINNER)
sweep = sorted([best] + [run_experiment(WINNER, r) for r in (4, 16, 32)], key=lambda r: r["rank"])
sweep_df = pd.DataFrame(sweep)[["rank", "trainable_params", "peak_train_mem_gb", "train_minutes", "tool_accuracy_pct",
                                "full_match_pct", "json_valid_pct"]]
sweep_df.to_csv(RESULTS / "rank_sweep.csv", index=False)
(RESULTS / "winner.json").write_text(json.dumps({"winner": WINNER}))
sweep_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(sweep_df["rank"], sweep_df["tool_accuracy_pct"], marker="o", label="Tool accuracy")
ax.plot(sweep_df["rank"], sweep_df["full_match_pct"], marker="s", linestyle="--", label="Full match")
for x, y in zip(sweep_df["rank"], sweep_df["tool_accuracy_pct"]):
    ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
ax.set_xscale("log", base=2)
ax.set_xticks(sweep_df["rank"], [str(r) for r in sweep_df["rank"]])
ax.set_xlabel("LoRA rank r (log scale)")
ax.set_ylabel("% of 200 test examples")
ax.set_title(f"{WINNER.upper()}: accuracy vs rank")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "rank_sweep.png", dpi=200)
plt.show()

## Failure cases of the best fine-tuned model (for the error analysis)

In [ ]:
all_runs = main_results + sweep
best_run = max(all_runs, key=lambda r: (r["full_match"], r["tool_correct"]))["run"]
preds = [json.loads(line) for line in open(RESULTS / f"predictions_{best_run}.jsonl")]
failures = [p for p in preds if not p["full_match"] or not p["json_valid"]]
by_label = pd.Series([f["label"] for f in failures]).value_counts().to_dict()
(RESULTS / "failures.json").write_text(json.dumps({"best_run": best_run, "label_counts": by_label, "failures": failures}, indent=2))
print(best_run, "-", len(failures), "failures:", by_label)
for f in failures[:5]:
    print(f"\n[{f['label']}] {f['user'][:200]}\n  gold: {json.dumps(f['gold'])}\n  got : {f['output'][:300]}")